# Phase 2 — Pseudo-labels from reports

Hybrid report labeling (rules + optional LLM) → train ConvNeXt-Tiny MIL on 4,407 studies.

- OOF metric computed on **58 labeled** studies only (compare to Phase 1 baseline 0.5295).
- Test inference is **image-only** (no reports).
- Pretrained backbone: set `cfg["model"]["pretrained_weights"]` / `variant` in export script.
- Daily: Cursor → Kaggle Jupyter Server ([docs/KAGGLE.md](../docs/KAGGLE.md)).
- Submit: `python scripts/push_kaggle_kernel.py phase2` → Save & Run All.

## Setup

In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch


def _src() -> Path:
    work = Path("/kaggle/working")
    work_src = work / "src"
    mount = Path("/kaggle/input/rsna-knee-code")
    if work.is_dir() and not (work_src / "rsna_knee").is_dir() and (mount / "src" / "rsna_knee").is_dir():
        shutil.copytree(mount / "src", work_src)
        if (mount / "configs").is_dir():
            shutil.copytree(mount / "configs", work / "configs", dirs_exist_ok=True)
    if (work_src / "rsna_knee").is_dir():
        return work_src
    for root in (Path.cwd(), Path.cwd().parent):
        src = root / "src"
        if (src / "rsna_knee").is_dir():
            return src
    raise FileNotFoundError("Run: python scripts/sync_code_to_jupyter.py")


_SRC = _src()
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))
REPO_ROOT = _SRC.parent

from rsna_knee.constants import TARGET_LABELS
from rsna_knee.data import load_test_table, predictions_to_submission
from rsna_knee.data.schema import labels_present_mask, load_train_table
from rsna_knee.models.weights import resolve_pretrained_weights
from rsna_knee.reports import evaluate_labeler, generate_pseudo_labels, report_eda_summary
from rsna_knee.reports.rules import RuleLabeler
from rsna_knee.training import macro_roc_auc, predict_test_ensemble, run_phase2_training
from rsna_knee.utils.config import load_config
from rsna_knee.utils.paths import default_data_root, is_kaggle_kernel

ON_KAGGLE = is_kaggle_kernel()
CFG_NAME = "kaggle_phase2" if ON_KAGGLE else "phase2"
cfg = load_config(CFG_NAME)
DATA_ROOT = default_data_root()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR = Path(cfg["paths"]["output_dir"])
if not OUT_DIR.is_absolute():
    OUT_DIR = REPO_ROOT / OUT_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Environment : {'Kaggle' if ON_KAGGLE else 'local'}")
print(f"Data root   : {DATA_ROOT}")
print(f"Config      : {CFG_NAME}")

## 1. Report EDA

In [ ]:
eda = report_eda_summary(DATA_ROOT)
print(json.dumps(eda, indent=2))
display(pd.DataFrame(eda["language_distribution"]))
display(pd.DataFrame(eda["keyword_hit_rates_labeled"]))

## 2. Validate rule labeler on 58 labeled studies

In [ ]:
rule_metrics, rule_macro_f1 = evaluate_labeler(RuleLabeler(), DATA_ROOT)
print(f"Rule labeler macro F1: {rule_macro_f1:.3f}")
display(rule_metrics.sort_values("f1", ascending=False))

## 3. Generate hybrid pseudo-labels

In [ ]:
pseudo_path = Path(cfg["data"]["pseudo_labels_path"])
if not pseudo_path.is_absolute():
    pseudo_path = REPO_ROOT / pseudo_path
pseudo_path.parent.mkdir(parents=True, exist_ok=True)

llm_path = cfg.get("reports", {}).get("llm_model_path")
pseudo_df = generate_pseudo_labels(
    DATA_ROOT,
    output_path=pseudo_path,
    llm_model_path=llm_path,
    route_threshold=float(cfg.get("reports", {}).get("route_threshold", 0.7)),
)
print(f"Wrote {pseudo_path}  shape={pseudo_df.shape}")
display(pseudo_df.head())

## 4. Resolve pretrained weights

In [ ]:
weight_name = cfg["model"].get("pretrained_weights", "convnext_tiny_imagenet.pth")
variant = cfg["model"].get("pretrained_variant")
pretrained_path = resolve_pretrained_weights(
    filename=weight_name,
    variant=variant,
    allow_missing=bool(cfg["model"].get("allow_random_init", False)),
)
print(f"Pretrained weights: {pretrained_path}")

## 5. Train on pseudo-labels (OOF on 58 labeled)

In [ ]:
CKPT_DIR = Path(cfg["paths"]["checkpoint_dir"])
if not CKPT_DIR.is_absolute():
    CKPT_DIR = REPO_ROOT / CKPT_DIR
CKPT_DIR.mkdir(parents=True, exist_ok=True)

volume_shape = tuple(cfg["data"]["volume_shape"])
max_series = int(cfg["data"]["max_series"])
MAX_EPOCHS = int(cfg["training"]["max_epochs"])
if not ON_KAGGLE and DEVICE.type == "cpu":
    MAX_EPOCHS = min(MAX_EPOCHS, 1)

train_result = run_phase2_training(
    DATA_ROOT,
    n_folds=int(cfg["training"]["n_folds"]),
    max_epochs=MAX_EPOCHS,
    batch_size=int(cfg["training"]["batch_size"]),
    learning_rate=float(cfg["training"]["learning_rate"]),
    volume_shape=volume_shape,
    max_series=max_series,
    checkpoint_dir=CKPT_DIR,
    seed=int(cfg["seed"]),
    pretrained_path=pretrained_path,
    allow_random_init=bool(cfg["model"].get("allow_random_init", False)),
    tta=bool(cfg["inference"].get("tta", False)),
    num_workers=int(cfg["data"].get("num_workers", 0)),
    model_name=cfg["model"]["name"],
    use_amp=bool(cfg["training"].get("amp", True)),
    gpu_cache=bool(cfg["training"].get("gpu_cache", False)),
    data_parallel=bool(cfg["training"].get("data_parallel", False)),
    slice_chunk=int(cfg["model"].get("slice_chunk", 16)),
    labeled_only=bool(cfg["data"].get("labeled_only", False)),
    pseudo_labels_path=pseudo_path,
    min_confidence=float(cfg["data"].get("min_confidence", 0.7)),
    confidence_weighted_loss=bool(cfg["training"].get("confidence_weighted_loss", True)),
    eval_labeled_only=bool(cfg["training"].get("eval_labeled_only", True)),
    mixup_alpha=float(cfg["training"].get("mixup_alpha", 0.4)),
    use_multimodal=bool(cfg["model"].get("use_multimodal", False)),
    text_model_path=cfg["model"].get("text_model_path"),
)

print(f"OOF macro ROC-AUC (58 labeled): {train_result['overall_auc']:.4f}")
print("Phase 1 baseline (from-scratch, 58 only): 0.5295")

## 6. Per-label OOF detail

In [ ]:
train = load_train_table(DATA_ROOT)
labeled_mask = labels_present_mask(train)
labeled_ids = set(train.loc[labeled_mask, "StudyInstanceUID"].astype(str))
labeled_idx = [i for i, uid in enumerate(train_result["study_ids"]) if uid in labeled_ids]

oof_preds = train_result["oof_preds"][labeled_idx]
oof_labels = train_result["oof_labels"][labeled_idx]
print("OOF macro ROC-AUC:", macro_roc_auc(oof_labels, oof_preds))

from sklearn.metrics import roc_auc_score
rows = []
for i, name in enumerate(TARGET_LABELS):
    yt = oof_labels[:, i]
    if len(np.unique(yt)) < 2:
        rows.append({"label": name, "auc": float("nan")})
        continue
    rows.append({"label": name, "auc": float(roc_auc_score(yt, oof_preds[:, i]))})
display(pd.DataFrame(rows).sort_values("auc", ascending=False))

## 7. Test inference → submission.csv (image-only)

In [ ]:
ckpt_paths = [fr.checkpoint_path for fr in train_result["fold_results"]]
study_ids, test_preds = predict_test_ensemble(
    ckpt_paths,
    DATA_ROOT,
    volume_shape=volume_shape,
    max_series=max_series,
    batch_size=int(cfg["training"]["batch_size"]),
    tta=bool(cfg["inference"].get("tta", False)),
    allow_random_init=True,
    model_name=cfg["model"]["name"],
    num_workers=int(cfg["data"].get("num_workers", 0)),
    use_amp=bool(cfg["training"].get("amp", True)),
    slice_chunk=int(cfg["model"].get("slice_chunk", 16)),
)

sub = predictions_to_submission(study_ids, pd.DataFrame(test_preds, columns=TARGET_LABELS))
sub_path = OUT_DIR / "submission.csv"
sub.to_csv(sub_path, index=False)
print(f"Wrote {sub_path}  shape={sub.shape}")
display(sub.head())

## 8. Optional — multimodal experiment

Set `model.use_multimodal: true` and attach a clinical text encoder Dataset. Re-run section 5 with fusion at train time; inference remains image-only.